# Football EDA: Physical Performance Mini Case Study

This notebook analyzes GPS wearable tracking data from a training session to extract physical and movement insights, adapting the analytical framework to continuous physiological metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully \u2713')

## 1 · Load & Prepare Data

The raw wearable CSV contains sensor readings at ~10 Hz. Several channels (Speed, Heart Rate, Acceleration) are prone to hardware spikes that must be **clipped** to physiologically plausible ranges before any analysis. GPS coordinates are converted from degrees to a local metre-based frame so that pitch-level distances are meaningful.

In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv('dataset.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

# ── Elapsed time scaled to match session window ─────────────────────────────
df['elapsed_min'] = np.linspace(0, 4.1, len(df))

# ── Clean / Normalize sensor spikes ────────────────────────────────────────
# Wearable sensors frequently produce outlier spikes that are physically
# impossible. Clipping constrains every reading to a plausible physiological
# range, removing artefacts without discarding the entire row.
df['Speed_clean'] = df['Speed (km/h)'].clip(0, 35)
df['HR_clean']    = df['Heart Rate (BPM)'].clip(90, 210)
df['Accel_clean'] = df['Acceleration (m/s²)'].clip(0, 20)

# ── GPS — keep only valid coordinates & convert to metres ─────────────────
geo = df[(df['Latitude'] != 0) & (df['Longitude'] != 0)].copy()
lat0 = geo['Latitude'].mean()
lon0 = geo['Longitude'].mean()
geo['x_m'] = (geo['Longitude'] - lon0) * np.cos(np.radians(lat0)) * 111320
geo['y_m'] = (geo['Latitude']  - lat0) * 111320

# ── Convenience arrays ────────────────────────────────────────────────────
t         = df['elapsed_min'].values
spd       = df['Speed_clean'].values
hr        = df['HR_clean'].values
accel     = df['Accel_clean'].values
pl_cum    = df['PlayerLoad (cumulative)'].values
pl_ins    = df['PlayerLoad (instant)'].values.clip(0, 0.07)
rot       = df['Body Rotation (°/s)'].clip(0, 85).values

print(f'Rows: {len(df):,}  |  GPS rows: {len(geo):,}  |  Duration: {t[-1]:.2f} min')
df[['Speed_clean','HR_clean','Accel_clean','PlayerLoad (instant)','Body Rotation (°/s)']].describe().round(2)

## 2 · Global Style Helpers

All plots use a consistent dark theme for visual consistency.

In [ ]:
# ── Colours ───────────────────────────────────────────────────────────────
DARK_BG  = '#010102'
PANEL_BG = '#111125'
TEXT_COL = '#e0e0e0'
GRID_COL = '#2a2a4a'

zone_order  = ['Walking', 'Jogging', 'Running', 'High-speed Running', 'Sprinting']
zone_colors = {
    'Walking':            '#00BFFF',
    'Jogging':            '#00CC44',
    'Running':            '#FFAA00',
    'High-speed Running': '#FF4400',
    'Sprinting':          '#FF0066'
}

def style_ax(ax, xlabel='', ylabel='', title=''):
    """Apply dark-theme styling to a single Axes."""
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, labelsize=8)
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.5, alpha=0.6)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_COL, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_COL, fontsize=9)
    if title:  ax.set_title(title,   color=TEXT_COL, fontsize=10, pad=6)

print('Style helpers defined \u2713')

---
### Plot 1: Spatial Movement Heatmap (Pitch Coverage)

This spatial visualization maps the player's GPS track coloured by speed zone, along with a 2-D position density heatmap. By converting raw Latitude/Longitude into a local metre frame, we can see actual on-pitch distances. The left panel shows sprint bursts concentrated along specific corridors, while the heatmap reveals the player's dominant operating zone. However, without the ball's location or opponent positions, this raw movement data lacks the tactical context needed to evaluate defensive/offensive effectiveness.

In [ ]:
# Plot 1: GPS Positional Data — Pitch Coverage & Heatmap
fig1, (ax_track, ax_heat) = plt.subplots(1, 2, figsize=(16, 5), facecolor=DARK_BG)
fig1.suptitle('GPS Positional Data — Pitch Coverage', color=TEXT_COL, fontsize=13, y=1.01)

# ── Left: track coloured by speed zone ─────────────────────────────────────
for zone in zone_order:
    zdf = geo[geo['Speed Zone'] == zone]
    ax_track.scatter(zdf['x_m'], zdf['y_m'], s=0.8,
                     color=zone_colors[zone], label=zone, alpha=0.8, linewidths=0)

ax_track.scatter(geo['x_m'].iloc[0],  geo['y_m'].iloc[0],
                 marker='^', color='#00FF88', s=100, zorder=10)
ax_track.scatter(geo['x_m'].iloc[-1], geo['y_m'].iloc[-1],
                 marker='s', color='#FF4444', s=100, zorder=10)

patches = [mpatches.Patch(color=zone_colors[z], label=z) for z in zone_order]
patches += [Line2D([0],[0], marker='^', color='w', markerfacecolor='#00FF88', markersize=8, label='Start'),
            Line2D([0],[0], marker='s', color='w', markerfacecolor='#FF4444', markersize=8, label='End')]
ax_track.legend(handles=patches, fontsize=7, loc='upper left',
                facecolor='#1a1a2e', edgecolor=GRID_COL, labelcolor=TEXT_COL)
style_ax(ax_track, xlabel='x (m)', ylabel='y (m)', title='Track coloured by Speed Zone')

# ── Right: 2-D position heatmap ───────────────────────────────────────────
heat_cmap = LinearSegmentedColormap.from_list(
    'heat', ['#000022', '#0000ff', '#00ffff', '#ffff00', '#ff0000'])
h, xedg, yedg, img = ax_heat.hist2d(geo['x_m'], geo['y_m'], bins=80, cmap=heat_cmap)
cb = fig1.colorbar(img, ax=ax_heat, pad=0.02)
cb.ax.tick_params(colors=TEXT_COL, labelsize=7)
cb.set_label('Sample count', color=TEXT_COL, fontsize=8)

ax_heat.scatter(geo['x_m'].iloc[0],  geo['y_m'].iloc[0],  marker='^', color='#00FF88', s=80, zorder=10)
ax_heat.scatter(geo['x_m'].iloc[-1], geo['y_m'].iloc[-1], marker='s', color='#FF4444', s=80, zorder=10)
leg2 = [Line2D([0],[0], marker='^', color='w', markerfacecolor='#00FF88', markersize=8, label='Start'),
        Line2D([0],[0], marker='s', color='w', markerfacecolor='#FF4444', markersize=8, label='End')]
ax_heat.legend(handles=leg2, fontsize=7, loc='upper left',
               facecolor='#1a1a2e', edgecolor=GRID_COL, labelcolor=TEXT_COL)
style_ax(ax_heat, xlabel='x (m)', ylabel='y (m)', title='Position Heatmap')

plt.tight_layout()
plt.show()

---
### Plot 2: Speed & Heart Rate Over Time (Metabolic Control)

This dual-axis line plot overlays the player's **cleaned** Speed against their **cleaned** Heart Rate across the session duration. The normalization step is critical here — without clipping sensor spikes, the extreme artefact values (e.g. 200+ km/h speed) would compress the y-axis and obscure the real physiological patterns. After cleaning, we can observe the characteristic 'sawtooth' pattern: heart rate rises sharply during sprint bursts and partially recovers during walking/jogging intervals, revealing the metabolic cost and recovery dynamics of the session.

In [ ]:
# Plot 2: Speed & Heart Rate Over Time (using normalized data)
fig2, ax_spd = plt.subplots(figsize=(14, 4), facecolor=DARK_BG)
ax_hr = ax_spd.twinx()

ax_spd.fill_between(t, spd, color='#00BFFF', alpha=0.35, label='Speed (km/h)')
ax_spd.plot(t, spd, color='#00BFFF', linewidth=0.4)

ax_hr.plot(t, hr, color='#FF4466', linewidth=0.6, alpha=0.85, label='Heart Rate (BPM)')

style_ax(ax_spd, xlabel='Elapsed time (min)', ylabel='Speed (km/h)',
         title='Speed & Heart-Rate Profile (Normalized)')
ax_hr.set_ylabel('Heart Rate (BPM)', color='#FF4466', fontsize=9)
ax_hr.tick_params(colors='#FF4466', labelsize=8)
for spine in ax_hr.spines.values():
    spine.set_edgecolor(GRID_COL)

lines1, labels1 = ax_spd.get_legend_handles_labels()
lines2, labels2 = ax_hr.get_legend_handles_labels()
ax_spd.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper right',
              facecolor='#1a1a2e', edgecolor=GRID_COL, labelcolor=TEXT_COL)

plt.tight_layout()
plt.show()

---
### Plot 3: Speed Zone Distribution (Workload Network)

This bar chart categorises the physical workload into distinct Speed Zones (Walking, Jogging, Running, High-speed Running, Sprinting). By identifying the proportion of time spent in each intensity band, we can build a 'workload network' of the player's physical structure. This is analogous to a passing network — instead of ball circulation, we see energy circulation. It reveals whether the player is fulfilling positional responsibilities beyond basic distance metrics, and highlights the tactical periodisation of effort across the session.

In [ ]:
# Plot 3: Speed Zone Distribution
if 'Speed Zone' in df.columns:
    zone_percent = df['Speed Zone'].value_counts(normalize=True) * 100

    fig3, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK_BG)
    fig3.suptitle('Speed Zone Distribution', color=TEXT_COL, fontsize=13, y=1.01)

    # ── Pie chart ──
    available_zones = [z for z in zone_order if z in zone_percent.index]
    pie_data = zone_percent.reindex(available_zones)
    pie_colors = [zone_colors[z] for z in available_zones]
    ax_pie.pie(pie_data.values, labels=available_zones, autopct='%1.1f%%',
               colors=pie_colors, textprops={'color': TEXT_COL, 'fontsize': 8})
    ax_pie.set_facecolor(DARK_BG)
    ax_pie.set_title('Proportion by Zone', color=TEXT_COL, fontsize=10)

    # ── Bar chart ──
    zone_counts = df['Speed Zone'].value_counts().reindex(available_zones)
    bars = ax_bar.bar(available_zones, zone_counts.values, color=pie_colors, edgecolor='#333')
    for i, v in enumerate(zone_counts.values):
        ax_bar.text(i, v + max(zone_counts.values)*0.01, str(v), ha='center',
                    color=TEXT_COL, fontsize=8)
    style_ax(ax_bar, xlabel='Speed Zone', ylabel='Data Points (Time spent)',
             title='Count by Zone')
    ax_bar.tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print('Speed Zone column not found.')

---
### Plot 4: Heart Rate Under Stress (High-Intensity vs Recovery)

This boxplot contrasts the player's **normalised** heart rate distribution during high-speed movements versus slower recovery periods. Similar to analysing pass completion under pressure, this segments physiological data by the `under_pressure` equivalent — movement intensity. The cleaning step (clipping HR to 90–210 BPM) is essential; without it, artefact values as high as 289 BPM would distort the box-and-whisker comparison, masking genuine cardiovascular stress patterns.

In [ ]:
# Plot 4: Heart Rate Distribution by Intensity (using normalized HR)
if 'Speed Zone' in df.columns and 'HR_clean' in df.columns:
    high_intensity = ['Running', 'High-speed Running', 'Sprinting']
    df['Intensity'] = np.where(df['Speed Zone'].isin(high_intensity),
                               'High Intensity', 'Low Intensity / Recovery')

    fig4, (ax_box, ax_violin) = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK_BG)
    fig4.suptitle('Heart Rate Under Physical Stress (Normalized)', color=TEXT_COL, fontsize=13, y=1.01)

    # ── Box plot ──
    sns.boxplot(x='Intensity', y='HR_clean', data=df, hue='Intensity',
                palette='Set2', legend=False, ax=ax_box)
    style_ax(ax_box, xlabel='Movement Intensity', ylabel='Heart Rate (BPM)',
             title='Box Plot')

    # ── Violin plot ──
    sns.violinplot(x='Intensity', y='HR_clean', data=df, hue='Intensity',
                   palette='Set2', legend=False, inner='quartile', ax=ax_violin)
    style_ax(ax_violin, xlabel='Movement Intensity', ylabel='Heart Rate (BPM)',
             title='Violin Plot')

    plt.tight_layout()
    plt.show()
else:
    print('Required columns for intensity analysis not found.')

---
### Plot 5: Cumulative PlayerLoad & Acceleration (Fatigue Correlation)

This plot tracks `PlayerLoad (cumulative)` over elapsed time, which quantifies the mechanical stress on the player via tri-axial accelerometer data. The **normalised** acceleration channel (clipped to 0–20 m/s²) is shown alongside the cumulative curve. The linear trendline slope reveals the fatigue accumulation rate — steeper gradients indicate higher sustained mechanical strain, increasing injury risk. This is the physical performance analogue of the xG-vs-distance relationship: as cumulative load increases, the probability of fatigue-related performance decline rises non-linearly.

In [ ]:
# Plot 5: Cumulative PlayerLoad & Acceleration over Time (normalized)
fig5, ax_pl = plt.subplots(figsize=(14, 4), facecolor=DARK_BG)
ax_acc = ax_pl.twinx()

# Cumulative PlayerLoad
ax_pl.plot(t, pl_cum, color='#8e44ad', linewidth=1.5, label='PlayerLoad (cumulative)')

# Normalized acceleration as background
ax_acc.fill_between(t, accel, color='#FFaa00', alpha=0.25, label='Acceleration (norm, m/s²)')
ax_acc.plot(t, accel, color='#FFaa00', linewidth=0.3, alpha=0.5)

# Trendline on cumulative load
clean_mask = ~np.isnan(t) & ~np.isnan(pl_cum)
if clean_mask.sum() > 1:
    z = np.polyfit(t[clean_mask], pl_cum[clean_mask], 1)
    p = np.poly1d(z)
    ax_pl.plot(t, p(t), 'r--', alpha=0.8, linewidth=1.2,
               label=f'Trend (Slope: {z[0]:.3f})')

style_ax(ax_pl, xlabel='Elapsed time (min)', ylabel='PlayerLoad (cumulative)',
         title='Cumulative PlayerLoad & Acceleration (Fatigue)')
ax_acc.set_ylabel('Acceleration (m/s²)', color='#FFaa00', fontsize=9)
ax_acc.tick_params(colors='#FFaa00', labelsize=8)
for spine in ax_acc.spines.values():
    spine.set_edgecolor(GRID_COL)

lines1, labels1 = ax_pl.get_legend_handles_labels()
lines2, labels2 = ax_acc.get_legend_handles_labels()
ax_pl.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left',
             facecolor='#1a1a2e', edgecolor=GRID_COL, labelcolor=TEXT_COL)

plt.tight_layout()
plt.show()